In [1]:
!pip install -q transformers datasets evaluate seqeval peft accelerate bitsandbytes


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 16.4 MB/s eta 0:00:00


In [2]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from peft import LoraConfig, get_peft_model
import evaluate


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import os

data_path = "/content/drive/MyDrive/NLP_PROJECT/data/"

def read_conll(path):
    sentences, labels = [], []
    temp_tokens, temp_labels = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:  # End of sentence
                if temp_tokens:
                    sentences.append(temp_tokens)
                    labels.append(temp_labels)
                    temp_tokens, temp_labels = [], []
            else:
                splits = line.split()
                temp_tokens.append(splits[0])
                temp_labels.append(splits[-1])
    return sentences, labels

sentences, labels = read_conll(os.path.join(data_path, "dataset.txt"))

print("Total samples:", len(sentences))
print("First example:", sentences[0])
print("Labels:", labels[0])


Total samples: 19205
First example: ['-DOCSTART-']
Labels: ['O']


In [6]:
from sklearn.model_selection import train_test_split

train_sents, temp_sents, train_labels, temp_labels = train_test_split(
    sentences, labels, test_size=0.2, random_state=42
)
val_sents, test_sents, val_labels, test_labels = train_test_split(
    temp_sents, temp_labels, test_size=0.5, random_state=42
)

print("Train:", len(train_sents))
print("Validation:", len(val_sents))
print("Test:", len(test_sents))


Train: 15364
Validation: 1920
Test: 1921


In [7]:
#Sentence, Word & Character Counts (Train / Test)
def dataset_stats(sentences):
    num_sent = len(sentences)
    num_words = sum(len(s) for s in sentences)
    num_chars = sum(len(token) for s in sentences for token in s)
    return num_sent, num_words, num_chars

train_stats = dataset_stats(train_sents)
test_stats = dataset_stats(test_sents)

print("\n=== TABLE 3: Dataset Statistics ===")
print(f"{'Set':<10}{'Sentences':<12}{'Words':<12}{'Characters'}")
print(f"Train     {train_stats[0]:<12}{train_stats[1]:<12}{train_stats[2]}")
print(f"Test      {test_stats[0]:<12}{test_stats[1]:<12}{test_stats[2]}")



=== TABLE 3: Dataset Statistics ===
Set       Sentences   Words       Characters
Train     15364       343231      1501972
Test      1921        43304       190775


In [9]:
#Distribution of Major Entities

from collections import Counter

def count_entities(label_sequences):
    counter = Counter()
    for seq in label_sequences:
        for label in seq:
            if label.startswith("B-"):      # count only entity beginnings
                counter[label[2:]] += 1
    return counter

train_entity_counts = count_entities(train_labels)
test_entity_counts = count_entities(test_labels)

print("\n=== Named Entity Distribution ===")
print(f"{'Entity Type':<25}{'Train':<10}{'Test'}")

all_types = sorted(set(train_entity_counts.keys()) | set(test_entity_counts.keys()))
for t in all_types:
    print(f"{t:<25}{train_entity_counts.get(t, 0):<10}{test_entity_counts.get(t, 0)}")



=== Named Entity Distribution ===
Entity Type              Train     Test
AffectedPopulation       140       9
CollapsedStructure       81        7
Date                     3947      510
Death_And_Toll           512       39
Fire                     193       19
Floods                   1497      195
InfrastructureDamage     58        6
Location                 7453      1005
MissingPersons           6         1
NaturalHazards           3654      412
PowerOutage              51        5
RoadBlocked              20        4
WaterShortage            39        5
WaterShrotage            3         0


In [11]:
#Example of Tokenization & Labeling

def find_example():
    for s, l in zip(train_sents, train_labels):
        if any(lbl != "O" for lbl in l):
            return s, l

example_tokens, example_labels = find_example()

print("\n=== Example Tokenization and Labeling ===")
print(f"{'Token':<20}Label")
for tok, lab in zip(example_tokens, example_labels):
    print(f"{tok:<20}{lab}")



=== Example Tokenization and Labeling ===
Token               Label
-                   O
I                   O
believe             O
there               O
should              O
be                  O
consistent          O
standards           O
for                 O
flood               B-Floods
and                 O
coastal             O
resilience          O
in                  O
England             B-Location
.                   O


In [12]:
#Encode labels as integers, build mappings
from collections import defaultdict

# Collect all unique labels
all_labels = sorted(set([lab for seq in train_labels + val_labels + test_labels for lab in seq]))

# Create label dictionaries
label2idx = {label: idx for idx, label in enumerate(all_labels)}
idx2label = {idx: label for label, idx in label2idx.items()}

print("Label mapping:", label2idx)

# Convert string labels → integer IDs
def encode_labels(label_sequences, mapping):
    return [[mapping[label] for label in seq] for seq in label_sequences]

train_label_ids = encode_labels(train_labels, label2idx)
val_label_ids = encode_labels(val_labels, label2idx)
test_label_ids = encode_labels(test_labels, label2idx)

# Quick sanity check
print("Sample tokens:", train_sents[0])
print("Sample labels (str):", train_labels[0])
print("Sample labels (int):", train_label_ids[0])


Label mapping: {'.O': 0, 'B-AffectedPopulation': 1, 'B-CollapsedStructure': 2, 'B-Date': 3, 'B-Death_And_Toll': 4, 'B-Fire': 5, 'B-Floods': 6, 'B-InfrastructureDamage': 7, 'B-Location': 8, 'B-MissingPersons': 9, 'B-NaturalHazards': 10, 'B-PowerOutage': 11, 'B-RoadBlocked': 12, 'B-WaterShortage': 13, 'B-WaterShrotage': 14, 'I-AffectedPopulation': 15, 'I-CollapsedStructure': 16, 'I-Date': 17, 'I-Death_And_Toll': 18, 'I-Floods': 19, 'I-InfrastructureDamage': 20, 'I-Location': 21, 'I-MissingPersons': 22, 'I-NaturalHazards': 23, 'I-PowerOutage': 24, 'I-RoadBlocked': 25, 'I-WaterShortage': 26, 'I-WaterShrotage': 27, 'O': 28}
Sample tokens: ['-', 'I', 'believe', 'there', 'should', 'be', 'consistent', 'standards', 'for', 'flood', 'and', 'coastal', 'resilience', 'in', 'England', '.']
Sample labels (str): ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B-Floods', 'O', 'O', 'O', 'O', 'B-Location', 'O']
Sample labels (int): [28, 28, 28, 28, 28, 28, 28, 28, 28, 6, 28, 28, 28, 28, 8, 28]


In [21]:

from transformers import AutoTokenizer
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)


base_model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label2idx),
    id2label=idx2label,
    label2id=label2idx
)



# Apply LoRA
lora_config = LoraConfig(
    r=8,              # rank
    lora_alpha=32,    # scaling
    target_modules=["q_lin", "v_lin"],  # DistilBERT uses q_lin, v_lin for attention
    lora_dropout=0.1,
    bias="none",
    task_type="TOKEN_CLS"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 169,757 || all params: 66,554,938 || trainable%: 0.2551


In [27]:
# Create HuggingFace datasets
train_ds = Dataset.from_dict({"tokens": train_sents, "labels": train_label_ids})
val_ds = Dataset.from_dict({"tokens": val_sents, "labels": val_label_ids})
test_ds = Dataset.from_dict({"tokens": test_sents, "labels": test_label_ids})

# Tokenize the datasets
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_idx:
                label_ids.append(label[word_id])
            else:
                label_ids.append(-100)
            previous_word_idx = word_id
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_datasets = DatasetDict({
    "train": train_ds.map(tokenize_and_align_labels, batched=True),
    "validation": val_ds.map(tokenize_and_align_labels, batched=True),
    "test": test_ds.map(tokenize_and_align_labels, batched=True),
})

print(tokenized_datasets)

Map:   0%|          | 0/15364 [00:00<?, ? examples/s]

Map:   0%|          | 0/1920 [00:00<?, ? examples/s]

Map:   0%|          | 0/1921 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 15364
    })
    validation: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1920
    })
    test: Dataset({
        features: ['tokens', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 1921
    })
})


In [ ]:
!pip install --upgrade transformers

In [15]:
import transformers
print(transformers.__version__)

4.57.2


In [28]:

#Training setup for LoRA disaster NER

import numpy as np
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification
import evaluate
import torch

batch_size = 16

# --- Training arguments ---
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to=[]  # <-- Disable all integrations (wandb, tensorboard, etc.)
)



# --- Data collator ---
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# --- Metrics setup ---
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_preds = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):
        temp_preds = []
        temp_labels = []
        for p, l in zip(pred_seq, label_seq):
            if l != -100:  # ignore padding
                temp_preds.append(idx2label[p])
                temp_labels.append(idx2label[l])
        true_preds.append(temp_preds)
        true_labels.append(temp_labels)

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

# --- Initialize Trainer ---
trainer = Trainer(
    model=model,                         # LoRA-adapted model
    args=training_args,
    train_dataset=tokenized_datasets["train"], # Using tokenized dataset
    eval_dataset=tokenized_datasets["validation"], # Using tokenized dataset
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# --- Optional: inspect trainable parameters ---
model.print_trainable_parameters()


trainable params: 169,757 || all params: 66,554,938 || trainable%: 0.2551


/tmp/ipython-input-2323847564.py:61: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [29]:
print("\n===Hyperparameters Used ===")
print(f"{'Hyperparameter':<25}{'Value'}")
print(f"{'Epochs':<25}{training_args.num_train_epochs}")
print(f"{'Learning Rate':<25}{training_args.learning_rate}")
print(f"{'Batch Size':<25}{training_args.per_device_train_batch_size}")
print(f"{'Optimizer':<25}{'AdamW (default in Transformers)'}")
print(f"{'LoRA r':<25}{lora_config.r}")
print(f"{'LoRA alpha':<25}{lora_config.lora_alpha}")
print(f"{'LoRA dropout':<25}{lora_config.lora_dropout}")



===Hyperparameters Used ===
Hyperparameter           Value
Epochs                   3
Learning Rate            5e-05
Batch Size               16
Optimizer                AdamW (default in Transformers)
LoRA r                   8
LoRA alpha               32
LoRA dropout             0.1


In [30]:
# Initialize the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

In [31]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/tmp/ipython-input-3910173894.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.682200,0.188051,0.604071,0.638560,0.620837,0.954765
2,0.175800,0.152548,0.653436,0.672376,0.662771,0.959940
3,0.151600,0.146144,0.658577,0.703118,0.680119,0.962273


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: .O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: .O seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/se

TrainOutput(global_step=2883, training_loss=0.2613070061584748, metrics={'train_runtime': 185.2328, 'train_samples_per_second': 248.833, 'train_steps_per_second': 15.564, 'total_flos': 733211384839056.0, 'train_loss': 0.2613070061584748, 'epoch': 3.0})

In [32]:
metrics = trainer.evaluate(tokenized_datasets["test"])
print(metrics)


{'eval_loss': 0.12848719954490662, 'eval_precision': 0.6427425821972734, 'eval_recall': 0.7185118780815778, 'eval_f1': 0.6785185185185185, 'eval_accuracy': 0.9648300387954923, 'eval_runtime': 4.249, 'eval_samples_per_second': 452.108, 'eval_steps_per_second': 28.477, 'epoch': 3.0}


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
